## Example 1: the two propagator entry points, `xray` and `xray-lambda`

In [ ]:
import importlib.metadata as md
from opentelemetry.propagators.aws import AwsXRayLambdaPropagator, AwsXRayPropagator

# Example 1: the SDK builds its propagators from OTEL_PROPAGATORS by looking names up in the `opentelemetry_propagator`
# entry-point group. This distribution must register exactly `xray` and `xray-lambda`, each loading its class.
def norm(name):
    return name.lower().replace("_", "-")

ours = {ep.name: ep for ep in md.entry_points(group="opentelemetry_propagator")
        if norm(ep.dist.name) == "opentelemetry-propagator-aws-xray"}
print(md.version("opentelemetry-propagator-aws-xray"), {name: ep.value for name, ep in ours.items()})
assert sorted(ours) == ["xray", "xray-lambda"], sorted(ours)
assert ours["xray"].load() is AwsXRayPropagator and ours["xray-lambda"].load() is AwsXRayLambdaPropagator
assert issubclass(AwsXRayLambdaPropagator, AwsXRayPropagator)
# The one header it reads and writes.
assert AwsXRayPropagator().fields == {"X-Amzn-Trace-Id"}, AwsXRayPropagator().fields

## Example 2: extract an X-Ray trace header into an OpenTelemetry span context

In [ ]:
from opentelemetry import trace
from opentelemetry.propagators.aws import AwsXRayPropagator

# Example 2: the header's Root carries the 128-bit trace id (version 1, an 8-hex time part, a 24-hex random part),
# Parent the caller's 64-bit span id, Sampled the sampling decision. The extracted context is marked remote.
header = "Root=1-5759e988-bd862e3fe1be46a994272793;Parent=53995c3f42cd8ad8;Sampled=1"
context = AwsXRayPropagator().extract({"X-Amzn-Trace-Id": header})
span_context = trace.get_current_span(context).get_span_context()
print(f"trace_id={span_context.trace_id:032x} span_id={span_context.span_id:016x} flags={span_context.trace_flags}")
assert span_context.is_valid and span_context.is_remote
assert span_context.trace_id == 0x5759E988BD862E3FE1BE46A994272793
assert span_context.span_id == 0x53995C3F42CD8AD8
assert span_context.trace_flags.sampled
# The order of the key=value pairs does not matter, and an unknown key (here Self, set by some AWS services) is ignored.
shuffled = "Sampled=1;Self=1-67891233-12456789abcdef012345678;Parent=53995c3f42cd8ad8;Root=1-5759e988-bd862e3fe1be46a994272793"
again = trace.get_current_span(AwsXRayPropagator().extract({"X-Amzn-Trace-Id": shuffled})).get_span_context()
assert (again.trace_id, again.span_id, again.trace_flags) == (span_context.trace_id, span_context.span_id, span_context.trace_flags)

## Example 3: inject a live span into a header, and read it back (round trip)

## Example 4: malformed headers are refused (no parent), and Sampled=0 is kept as not sampled

In [ ]:
from opentelemetry import trace
from opentelemetry.propagators.aws import AwsXRayPropagator

# Example 4: a header that breaks the format must not produce a parent: the propagator logs it (at debug level) and returns the context
# it was given. Each bad case changes ONE thing in a good header.
good = "Root=1-5759e988-bd862e3fe1be46a994272793;Parent=53995c3f42cd8ad8;Sampled=1"
def extracted(header):
    return trace.get_current_span(AwsXRayPropagator().extract({"X-Amzn-Trace-Id": header})).get_span_context()

assert extracted(good).is_valid  # POSITIVE CONTROL for the cases below
bad = {
    "trace id version 2": good.replace("Root=1-", "Root=2-"),
    "short random part": good.replace("bd862e3fe1be46a994272793", "bd862e3fe1be46a99427279"),
    "non-hex parent": good.replace("53995c3f42cd8ad8", "53995c3f42cd8adz"),
    "sampled flag 2": good.replace("Sampled=1", "Sampled=2"),
    "no parent": "Root=1-5759e988-bd862e3fe1be46a994272793;Sampled=1",
}
results = {name: extracted(header).is_valid for name, header in bad.items()}
print(results)
assert not any(results.values()), results
not_sampled = extracted(good.replace("Sampled=1", "Sampled=0"))
assert not_sampled.is_valid and not not_sampled.trace_flags.sampled

## Example 5: `OTEL_PROPAGATORS=xray` selects it globally, and `xray-lambda` reads Lambda's `_X_AMZN_TRACE_ID`

In [ ]:
import os, subprocess, sys

# Example 5: the global propagator is built from OTEL_PROPAGATORS when `opentelemetry.propagate` is first imported, so
# each setting runs in a fresh interpreter. `xray-lambda` falls back to the `_X_AMZN_TRACE_ID` variable Lambda sets for
# each invocation when the carrier has no header.
probe = ("from opentelemetry import propagate, trace\n"
         "textmap = propagate.get_global_textmap()\n"
         "print(sorted(type(p).__name__ for p in textmap._propagators))\n"
         "sc = trace.get_current_span(propagate.extract({})).get_span_context()\n"
         "print(sc.is_valid, format(sc.trace_id, '032x'))\n")
header = "Root=1-5759e988-bd862e3fe1be46a994272793;Parent=53995c3f42cd8ad8;Sampled=1"
def run(propagators):
    env = {**os.environ, "OTEL_PROPAGATORS": propagators, "_X_AMZN_TRACE_ID": header}
    done = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True, timeout=120)
    assert done.returncode == 0, done.stderr[-2000:]
    return done.stdout.splitlines()
xray, lam, w3c = run("xray"), run("xray-lambda"), run("tracecontext,baggage")
print(xray, lam, w3c, sep="\n")
assert xray == ["['AwsXRayPropagator']", "False 00000000000000000000000000000000"], xray
assert lam == ["['AwsXRayLambdaPropagator']", "True 5759e988bd862e3fe1be46a994272793"], lam
# NEGATIVE CONTROL: the default W3C propagators ignore the variable.
assert w3c[0] == "['TraceContextTextMapPropagator', 'W3CBaggagePropagator']" and w3c[1].startswith("False"), w3c